In [ ]:
"""Project Overview
Business Objective:
To ensure there is no discrimination between employees, Company X seeks to
automate salary decisions by using historical hiring data. The aim is to build a
 model that predicts the salary to be offered to a candidate based on objective
 features like experience, education, certifications, and more, thereby
 minimizing manual judgment and promoting fairness.
 """

In [ ]:
%pip install numpy pandas matplotlib scikit-learn gdown

In [ ]:
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
#1.loading the data
import gdown

file_id = "18zfwFUrcPmQVUxpJdIodGxlncMgOPacE"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "expected_ctc.csv", quiet=False)

#analyzing t

df=pd.read_csv("expected_ctc.csv")
#analyzing the data for the columns
df.head()

In [ ]:
#analyzing the datatypes of the column
df.info()

In [ ]:
#analyzing the count of rows and columns
df.shape

In [ ]:
#2.checking for missing data
df.isna().sum()

In [ ]:
df["Current_CTC"].describe()

In [ ]:
#separating departments and currentctc
department = df["Department"]
current_ctc = df["Current_CTC"]

In [ ]:
dup = df.duplicated()
print((dup.sum()))
#finding duplicate columns

In [ ]:
#checking outliers
cont = df.select_dtypes(include=['number']).columns
plt.figure(figsize=(10, 10))
df[cont].boxplot(vert=False)
plt.title('With Outliers', fontsize=16)
plt.show()

In [ ]:
#fixing outliers
def remove_outlier(col):
    q1, q3 = np.percentile(col, [25, 75])
    iqr = q3 - q1
    lr = q1 - (1.5 * iqr)
    ur = q3 + (1.5 * iqr)
    return lr, ur

for column in df[cont].columns:
    lr, ur = remove_outlier(df[column])
    df[column] = np.where(df[column] > ur, ur, df[column])
    df[column] = np.where(df[column] < lr, lr, df[column])

In [ ]:
#after removing outliers
plt.figure(figsize=(10, 10))
df[cont].boxplot(vert=False)
plt.title('After Outlier Removal', fontsize=16)
plt.show()

In [ ]:
#comparing the department and its salaries
plt.style.use('dark_background')
bar1 = df.groupby("Department")["Current_CTC"].mean().sort_values(ascending=False)
plt.figure(figsize=(6, 6))
plt.bar(bar1.index, bar1.values, color='cyan')
plt.title("Average Current CTC by Department", fontsize=16)
plt.xlabel("Department", fontsize=12)
plt.ylabel("Average Current CTC", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#3.handling missing data
#fill missing numeric values with mean
for col in df.columns:
    if df[col].dtype != 'object':
        df[col] = df[col].fillna(df[col].mean())
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('Unknown')
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category').cat.codes
#filled missing categorical values with numeric values by assigning each unique
# element to a number

In [ ]:
""" I am going to use Random Forest method because it has high accuracy, helps to prevent overfitting
    and are relatively robust to outliers
    Salary decisions are often based on complex, non-linear combinations of features

    Random Forest can capture complex, non-linear interactions between these variables without requiring
    manual feature engineering.

    Random Forest can model these interactions automatically because it builds many decision trees that
    capture different splits in the data.

    This results in high generalization accuracy
"""

In [ ]:
#4.Training and testing
X = df.drop(columns=['Expected_CTC'])
Y = df['Expected_CTC']
#classifying the 75% of the dataset to training set and rest 25% for test set
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=42)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
Y_pred = rf.predict(X_test)

In [ ]:
mae = metrics.mean_absolute_error(Y_test, Y_pred)
rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
print(f"Random Forest MAE:  {mae:,.2f}")
print(f"Random Forest RMSE: {rmse:,.2f}")

In [ ]:
meanexpctc = df['Expected_CTC'].mean()
print(meanexpctc)

In [ ]:
#5.Evaluation
#comparing rmse and expectedctc
percentage = (rmse / meanexpctc) * 100
print(percentage)

In [ ]:
#comparing the predictions and testing data
plt.figure(figsize=(8, 6))
plt.scatter(test_labels, Y_pred, alpha=0.5, color='green')
plt.plot([test_labels.min(), test_labels.max()], [test_labels.min(), test_labels.max()], 'r--', lw=2)
#constructed a red line for the actual salaries
#this scatter plot explains actual salaries and the predicted salaries.
plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title('Actual vs Predicted Salary Scatter Plot')
plt.grid(True)
plt.show()

In [ ]:
print(
    f"Summary — RMSE: {rmse:,.2f}, Mean Expected_CTC: {meanexpctc:,.2f}, Relative error: {percentage:.2f}%"
)
print(
    "The goal of this project was to develop a machine learning model that predicts the salary "
    "to be offered to a candidate based on their profile, using historical data. "
    "This approach aims to reduce manual judgment and ensure fairness by providing consistent "
    "salary offers for employees with similar profiles. "
    "By using the Random Forest regression algorithm, the model achieved an exceptionally low "
    "error rate, meaning its predicted salaries are very close to the actual salaries."
)